# Figure 6 — Dynamic Rodeo filter circuit

**Paper location:** Appendix E.1, Fig. 6. **Requires Qiskit.**

## What this figure shows

The compiled, **measurement-conditioned** Rodeo circuit for the single-spin embedding
`M`, drawn for two cycles. The defining feature is the *dynamic* structure: after each
cycle's ancilla is measured, the controlled evolutions of every later cycle sit inside a
classically-controlled block that runs only if the previous success bits were zero — so
a failure short-circuits the rest of the schedule (this is the gate-level basis of the
restart advantage in Fig. 2).

Each cycle uses the **phase-symmetric** form: the controlled evolutions `U± = e^{±iM t/2}`
act on the two ancilla branches, so the success outcome 0 implements the real filter
factor `cos(M t/2)` with no residual eigenvalue-dependent phase.

## Model & construction

| | |
|---|---|
| **Embedding** | `M = [[0,L],[L†,0]]` on 3 data qubits (row, col, branch) + 1 reused ancilla |
| **Schedule** | deterministic van der Corput times `{t_ℓ}` |
| **Per cycle** | `H` on ancilla → controlled `U±` → `H` → mid-circuit measurement |
| **Dynamic** | later cycles nested in a conditional on the earlier success bit |


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))
import numpy as np
import rodeo_ness as rn
from scipy.linalg import expm
try:
    import qiskit
    from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
    from qiskit.circuit.library import UnitaryGate
    HAVE_QISKIT = True
    print("Qiskit", qiskit.__version__)
except Exception as e:
    HAVE_QISKIT = False
    print("Qiskit not available:", e)


Qiskit 2.4.2


### Build and draw the two-cycle dynamic circuit

In [2]:
assert HAVE_QISKIT, "install qiskit (pip install 'rodeo_ness[circuit]')"
import matplotlib.pyplot as plt
M = rn.hermitian_embedding(rn.single_spin_liouvillian(h=0.5))
times = rn.vdc_schedule(2, 8.0)
def U_pm(t, sign):
    return UnitaryGate(expm(sign*1j*M*t/2), label=f'U{"+" if sign>0 else "-"}')
mreg=QuantumRegister(3,"m"); rreg=QuantumRegister(1,"r"); creg=ClassicalRegister(2,"meas")
qc=QuantumCircuit(mreg,rreg,creg)
qc.h(mreg[0]); qc.barrier(label="|xi>")
# --- cycle 1 ---
qc.h(rreg[0])
qc.append(U_pm(times[0],+1).control(1),[rreg[0]]+list(mreg))
qc.append(U_pm(times[0],-1).control(1),[rreg[0]]+list(mreg))
qc.h(rreg[0]); qc.measure(rreg[0],creg[0])
# --- cycle 2 nested in a conditional: runs ONLY if the first success bit is 0 ---
# this is what makes the filter measurement-conditioned -- a failed first cycle
# short-circuits the rest of the schedule (the gate-level basis of restart)
with qc.if_test((creg[0], 0)):
    qc.h(rreg[0])
    qc.append(U_pm(times[1],+1).control(1),[rreg[0]]+list(mreg))
    qc.append(U_pm(times[1],-1).control(1),[rreg[0]]+list(mreg))
    qc.h(rreg[0])
qc.measure(rreg[0],creg[1])
fig=qc.draw("mpl",fold=20)
fig.savefig("rodeo_dynamic_circuit.pdf",bbox_inches="tight")
plt.show(); print(f"dynamic Rodeo circuit, depth {qc.depth()} -- cycle 2 nested in If(meas_0==0)")

dynamic Rodeo circuit, depth 8 -- cycle 2 nested in If(meas_0==0)
